# LC Route Dataset Generation For Improvement

This notebook generates LC route files for the synthetic raw graphs in `examples/data/raw_graphs_1000.pkl`. The output layout matches `examples/route_generator/lc_improvement_training.ipynb`:

- raw graphs: `examples/data/raw_graphs_1000.pkl`
- LC routes: `examples/lc_results/graph_XXXX/lc_lc_graph_XXXX_routes_routes.pkl`


## 1. Imports

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from hydra import compose, initialize_config_dir
from omegaconf import OmegaConf
from torch_geometric.loader import DataLoader

from connectpt.routes_generator.citygraph_dataset import get_dataset_from_config
from connectpt.routes_generator.utils import get_eval_cfg
from connectpt.routes_generator.eval_route_generator import eval_model
from connectpt.routes_generator.bee_colony import bee_colony
from connectpt.routes_generator.torch_utils import dump_routes, get_batch_tensor_from_routes
from connectpt.routes_generator.utils import process_standard_experiment_cfg, test_method


## 2. Paths And Experiment Parameters

In [2]:
def find_repo_root(start: Path | None = None) -> Path:
    start = Path.cwd().resolve() if start is None else start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "connectpt").exists() and (candidate / "examples").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root from current working directory")

ROOT_DIR = find_repo_root()
CFG_DIR = ROOT_DIR / "connectpt" / "routes_generator" / "cfg"
BENCHMARK_DIR = ROOT_DIR / "examples" / "data" / "benchmark"
MODEL_WEIGHTS_PATH = ROOT_DIR / "examples" / "data" / "model_weights" / "inductive_random_graphs_weighted_connectivity.pt"
RAW_GRAPHS_PATH = ROOT_DIR / "examples" / "data" / "raw_graphs_1000.pkl"
LC_RESULTS_DIR = ROOT_DIR / "examples" / "lc_results"

CITY_NAME = "Mumford0"

BASE_N_ROUTES = 3
BASE_MIN_ROUTE_LEN = 2
BASE_MAX_ROUTE_LEN = 5

SEEDED_N_ROUTES = BASE_N_ROUTES +1
SEEDED_MIN_ROUTE_LEN = BASE_MIN_ROUTE_LEN
SEEDED_MAX_ROUTE_LEN = BASE_MAX_ROUTE_LEN + 3

LC_SAMPLES = 100
USE_NEURAL_BCO = False
BCO_N_BEES = 10
BCO_N_ITERATIONS = 20
BCO_N_TYPE1_BEES = 5

DEMAND_TIME_WEIGHT = 0.33
ROUTE_TIME_WEIGHT = 0.33
MEDIAN_CONNECTIVITY_WEIGHT = 0.33

print(f"ROOT_DIR: {ROOT_DIR}")
print(f"CFG_DIR: {CFG_DIR}")
print(f"MODEL_WEIGHTS_PATH exists: {MODEL_WEIGHTS_PATH.exists()}")
print(f"RAW_GRAPHS_PATH: {RAW_GRAPHS_PATH}")
print(f"LC_RESULTS_DIR:  {LC_RESULTS_DIR}")


ROOT_DIR: D:\PythonProjects\connectpt
CFG_DIR: D:\PythonProjects\connectpt\connectpt\routes_generator\cfg
MODEL_WEIGHTS_PATH exists: True
RAW_GRAPHS_PATH: D:\PythonProjects\connectpt\examples\data\raw_graphs_1000.pkl
LC_RESULTS_DIR:  D:\PythonProjects\connectpt\examples\lc_results


## 3. Load Benchmark Tensors

## 4. Helper Functions

In [3]:
graphs_list = pd.read_pickle(RAW_GRAPHS_PATH)
print(f"Loaded raw graphs: {len(graphs_list)} from {RAW_GRAPHS_PATH}")


Loaded raw graphs: 1000 from D:\PythonProjects\connectpt\examples\data\raw_graphs_1000.pkl


In [4]:
graphs_list

[CityGraphData(
   fixed_routes=[0, 50],
   street_adj=[50, 50],
   drive_times=[50, 50],
   nexts=[50, 50],
   demand=[50, 50],
   stop={
     pos=[50, 2],
     x=[50, 2],
   },
   (stop, street, stop)={
     edge_index=[2, 130],
     edge_attr=[130, 1],
   },
   (stop, demand, stop)={
     edge_index=[2, 2450],
     edge_attr=[2450, 2],
   }
 ),
 CityGraphData(
   fixed_routes=[0, 50],
   street_adj=[50, 50],
   drive_times=[50, 50],
   nexts=[50, 50],
   demand=[50, 50],
   stop={
     pos=[50, 2],
     x=[50, 2],
   },
   (stop, street, stop)={
     edge_index=[2, 292],
     edge_attr=[292, 1],
   },
   (stop, demand, stop)={
     edge_index=[2, 2450],
     edge_attr=[2450, 2],
   }
 ),
 CityGraphData(
   fixed_routes=[0, 50],
   street_adj=[50, 50],
   drive_times=[50, 50],
   nexts=[50, 50],
   demand=[50, 50],
   stop={
     pos=[50, 2],
     x=[50, 2],
   },
   (stop, street, stop)={
     edge_index=[2, 124],
     edge_attr=[124, 1],
   },
   (stop, demand, stop)={
     edge_in

In [5]:
def make_test_dataloader(dataset_cfg):
    dataset = get_dataset_from_config(dataset_cfg, tensors=input_tensors)
    return DataLoader(dataset, batch_size=1)

def build_lc_cfg(run_name: str, n_routes: int, min_route_len: int, max_route_len: int,
                  demand_weight: float = None, route_weight: float = None, connectivity_weight: float = None):
    dw = demand_weight if demand_weight is not None else DEMAND_TIME_WEIGHT
    rw = route_weight if route_weight is not None else ROUTE_TIME_WEIGHT
    cw = connectivity_weight if connectivity_weight is not None else MEDIAN_CONNECTIVITY_WEIGHT
    params = {
        "dataset_name": "tensor",
        "n_routes": n_routes,
        "min_route_len": min_route_len,
        "max_route_len": max_route_len,
        "run_name": run_name,
        "model_weights": str(MODEL_WEIGHTS_PATH),
    }
    cfg = get_eval_cfg(str(CFG_DIR), "eval_model_mumford", params)
    # Hydra caches global state after first compose() call, so weights passed via overrides
    # are ignored on subsequent calls. Patch the cfg object directly to bypass the cache.
    OmegaConf.update(cfg, "experiment.cost_function.kwargs.demand_time_weight", dw)
    OmegaConf.update(cfg, "experiment.cost_function.kwargs.route_time_weight", rw)
    OmegaConf.update(cfg, "experiment.cost_function.kwargs.median_connectivity_weight", cw)
    cfg.batch_size = 1
    return cfg

def build_bco_cfg(run_name: str, n_routes: int, min_route_len: int, max_route_len: int, use_neural_bees: bool = USE_NEURAL_BCO):
    base_cfg_name = "neural_bco_mumford" if use_neural_bees else "bco_mumford"
    overrides = [
        "+eval=mumford0",
        "++eval.dataset.type=tensor",
        f"++eval.n_routes={n_routes}",
        f"++eval.min_route_len={min_route_len}",
        f"++eval.max_route_len={max_route_len}",
        f"++experiment.cost_function.kwargs.demand_time_weight={DEMAND_TIME_WEIGHT}",
        f"++experiment.cost_function.kwargs.route_time_weight={ROUTE_TIME_WEIGHT}",
        f"++experiment.cost_function.kwargs.median_connectivity_weight={MEDIAN_CONNECTIVITY_WEIGHT}",
        f"++run_name={run_name}",
        f"++n_bees={BCO_N_BEES}",
        f"++n_iterations={BCO_N_ITERATIONS}",
        f"++n_type1_bees={BCO_N_TYPE1_BEES}",
    ]
    if use_neural_bees:
        overrides.append(f"+model.weights={MODEL_WEIGHTS_PATH}")
    with initialize_config_dir(config_dir=str(CFG_DIR), version_base=None):
        cfg = compose(config_name=base_cfg_name, overrides=overrides)
    cfg.batch_size = 1
    return cfg

def run_lc(cfg, init_routes=None):
    dataloader = make_test_dataloader(cfg.eval.dataset)
    device, run_name, _, cost_obj, model = process_standard_experiment_cfg(
        cfg,
        run_name_prefix="lc_",
        weights_required=True,
    )
    init_cfg = OmegaConf.create({"method": "tensor"}) if init_routes is not None else None
    _, unserved_demand, metrics, routes = eval_model(
        model,
        dataloader,
        cfg.eval,
        cost_obj,
        n_samples=LC_SAMPLES,
        return_routes=True,
        silent=True,
        device=device,
        init_cfg=init_cfg,
        routes_tensor=init_routes,
    )
    return run_name, metrics, unserved_demand, routes

def run_bco(cfg, init_routes):
    dataloader = make_test_dataloader(cfg.eval.dataset)
    use_neural_bees = cfg.get("neural_bees", False)
    prefix = "neural_bco_" if use_neural_bees else "bco_"
    device, run_name, _, cost_obj, bee_model = process_standard_experiment_cfg(
        cfg,
        run_name_prefix=prefix,
        weights_required=use_neural_bees,
    )
    force_linking_unlinked = cfg.get("force_linking_unlinked", False)
    if not use_neural_bees:
        bee_model = None
    elif bee_model is not None:
        bee_model.force_linking_unlinked = force_linking_unlinked
        bee_model.eval()

    output = test_method(
        bee_colony,
        dataloader,
        cfg.eval,
        OmegaConf.create({"method": "tensor"}),
        cost_obj,
        silent=True,
        device=device,
        return_routes=True,
        routes_tensor=init_routes,
        n_bees=cfg.n_bees,
        n_iterations=cfg.n_iterations,
        n_type1_bees=cfg.get("n_type1_bees", None),
        n_type2_bees=cfg.get("n_type2_bees", None),
        bee_model=bee_model,
        force_linking_unlinked=force_linking_unlinked,
        adjustment_degree_weight=cfg.get("adjustment_degree_weight", 0.0),
        adjustment_degree_gap=cfg.get("adjustment_degree_gap", 0.1),
        adjustment_degree_mode=cfg.get("adjustment_degree_mode", "current"),
        adjustment_degree_objective=cfg.get("adjustment_degree_objective", "raw"),
        adjustment_degree_target=cfg.get("adjustment_degree_target", 0.2),
    )
    _, _, unserved_demand, metrics, routes = output
    return run_name, metrics, unserved_demand, routes

def summarize_run(label: str, metrics: dict, routes: torch.Tensor):
    n_nonempty_routes = int(((routes > -1).sum(dim=-1) > 1).sum().item())
    return {
        "method": label,
        "cost": float(metrics["cost"].item()),
        "ATT": float(metrics["ATT"].item()),
        "RTT": float(metrics["RTT"].item()),
        "$d_{un}$": float(metrics["$d_{un}$"].item()),
        "# routes": n_nonempty_routes,
    }


## 5. Baseline LC On Mumford0

In [6]:
# Default tensor dataset source; the generation loop replaces this with one graph dict per run.
input_tensors = RAW_GRAPHS_PATH


In [7]:
from tqdm import tqdm
import os

GROUP_PARAMS = [
    {"n_routes": 6,  "min_route_len": 2,  "max_route_len": 7},
    {"n_routes": 8,  "min_route_len": 4,  "max_route_len": 9},
    {"n_routes": 10, "min_route_len": 8,  "max_route_len": 12},
    {"n_routes": 12, "min_route_len": 10, "max_route_len": 14},
]
GROUP_SIZE = 250

# Predefined weight combinations (demand, route, connectivity), each sums to 1.0
WEIGHT_COMBINATIONS = [
    # pure
    (1.0, 0.0, 0.0),
    (0.0, 1.0, 0.0),
    (0.0, 0.0, 1.0),
    # two-component 80/20
    (0.8, 0.2, 0.0),
    (0.2, 0.8, 0.0),
    (0.8, 0.0, 0.2),
    (0.2, 0.0, 0.8),
    (0.0, 0.8, 0.2),
    (0.0, 0.2, 0.8),
    # two-component 50/50
    (0.5, 0.5, 0.0),
    (0.5, 0.0, 0.5),
    (0.0, 0.5, 0.5),
    # three-component dominant
    (0.6, 0.2, 0.2),
    (0.2, 0.6, 0.2),
    (0.2, 0.2, 0.6),
    # three-component 50/25/25
    (0.5, 0.25, 0.25),
    (0.25, 0.5, 0.25),
    (0.25, 0.25, 0.5),
    # three-component skewed
    (0.6, 0.3, 0.1),
    (0.6, 0.1, 0.3),
    (0.3, 0.6, 0.1),
    (0.1, 0.6, 0.3),
    (0.3, 0.1, 0.6),
    (0.1, 0.3, 0.6),
    # equal
    (1/3, 1/3, 1/3),
]

OUTPUT_BASE_DIR = LC_RESULTS_DIR
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)
print(f"LC route outputs will be written to: {OUTPUT_BASE_DIR}")

all_results = []

for i, g in enumerate(tqdm(graphs_list, desc="Processing graphs")):
    try:
        street_adj = g.street_adj
        demand = g.demand
        node_locs = g["stop"]["pos"]
        input_tensors = {
            "street_adj": street_adj,
            "demand": demand,
            "node_locs": node_locs,
        }

        group_idx = min(i // GROUP_SIZE, 3)
        group = GROUP_PARAMS[group_idx]

        demand_w, route_w, conn_w = WEIGHT_COMBINATIONS[i % len(WEIGHT_COMBINATIONS)]

        graph_run_name = f"lc_graph_{i:04d}"

        cfg_lc_base = build_lc_cfg(
            run_name=graph_run_name,
            n_routes=group["n_routes"],
            min_route_len=group["min_route_len"],
            max_route_len=group["max_route_len"],
            demand_weight=demand_w,
            route_weight=route_w,
            connectivity_weight=conn_w,
        )

        lc_base_run_name, lc_base_metrics, lc_base_unserved, lc_base_routes = run_lc(cfg_lc_base)

        graph_output_dir = OUTPUT_BASE_DIR / f"graph_{i:04d}"
        graph_output_dir.mkdir(parents=True, exist_ok=True)

        dump_routes(
            f"{lc_base_run_name}_routes",
            lc_base_routes.cpu(),
            out_dir=graph_output_dir,
        )

        with open(graph_output_dir / "metrics.txt", 'w') as f:
            f.write(f"Graph index: {i}\n")
            f.write(f"Group: {group_idx} (graphs {group_idx * GROUP_SIZE}–{(group_idx + 1) * GROUP_SIZE - 1})\n")
            f.write(f"n_routes: {group['n_routes']}, min_route_len: {group['min_route_len']}, max_route_len: {group['max_route_len']}\n")
            f.write(f"weight_combo_index: {i % len(WEIGHT_COMBINATIONS)}\n")
            f.write(f"demand_time_weight: {demand_w:.4f}\n")
            f.write(f"route_time_weight: {route_w:.4f}\n")
            f.write(f"median_connectivity_weight: {conn_w:.4f}\n")
            f.write(f"weights_sum: {demand_w + route_w + conn_w:.4f}\n")
            f.write(f"Run name: {lc_base_run_name}\n")
            f.write(f"Number of nodes: {node_locs.shape[0]}\n")
            f.write(f"Cost: {lc_base_metrics['cost'].item():.4f}\n")
            f.write(f"ATT: {lc_base_metrics['ATT'].item():.4f}\n")
            f.write(f"RTT: {lc_base_metrics['RTT'].item():.4f}\n")
            f.write(f"Unserved demand: {lc_base_unserved.sum().item():.4f}\n")
            f.write(f"Routes shape: {tuple(lc_base_routes.shape)}\n")

        result = {
            "graph_index": i,
            "group": group_idx,
            "n_routes": group["n_routes"],
            "min_route_len": group["min_route_len"],
            "max_route_len": group["max_route_len"],
            "weight_combo_index": i % len(WEIGHT_COMBINATIONS),
            "demand_time_weight": demand_w,
            "route_time_weight": route_w,
            "median_connectivity_weight": conn_w,
            "run_name": lc_base_run_name,
            "cost": float(lc_base_metrics["cost"].item()),
            "ATT": float(lc_base_metrics["ATT"].item()),
            "RTT": float(lc_base_metrics["RTT"].item()),
            "unserved_demand": float(lc_base_unserved.sum().item()),
            "routes": lc_base_routes.cpu(),
            "n_nodes": node_locs.shape[0],
            "output_dir": str(graph_output_dir),
        }
        all_results.append(result)

    except Exception as e:
        print(f"Error on graph {i}: {e}")
        continue

summary_path = OUTPUT_BASE_DIR / "all_results_summary.csv"

if all_results:
    df_results = pd.DataFrame([
        {k: v for k, v in r.items() if k not in ("routes", "output_dir")}
        for r in all_results
    ])
    df_results.to_csv(summary_path, index=False)
    print(f"Summary saved to: {summary_path}")

    print("\n" + "=" * 60)
    print("СТАТИСТИКА ПО ВСЕМ ГРАФАМ")
    print("=" * 60)
    print(f"Всего обработано: {len(all_results)} графов")
    for g_idx, gp in enumerate(GROUP_PARAMS):
        g_df = df_results[df_results["group"] == g_idx]
        if g_df.empty:
            continue
        print(f"\nГруппа {g_idx} (n={gp['n_routes']}, len={gp['min_route_len']}–{gp['max_route_len']}): {len(g_df)} графов")
        print(f"  Cost: mean={g_df['cost'].mean():.4f} ± {g_df['cost'].std():.4f}")
        print(f"  ATT:  mean={g_df['ATT'].mean():.4f} ± {g_df['ATT'].std():.4f}")
        print(f"  RTT:  mean={g_df['RTT'].mean():.4f} ± {g_df['RTT'].std():.4f}")


LC route outputs will be written to: D:\PythonProjects\connectpt\examples\lc_results


Processing graphs:   0%|          | 0/1000 [00:00<?, ?it/s]D:\PythonProjects\connectpt\connectpt\routes_generator\utils.py:304: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\ReduceOps.cpp:1858.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)
Processing graphs:   6%|▌         | 60/1000 [09:12<2:24:12,  9.20s/it]


KeyboardInterrupt: 

## Validate Generated Files For LC Improvement


In [ ]:
from connectpt.routes_generator.improvement_learning import load_raw_graphs_and_lc_routes

generated_graphs, generated_seed_routes = load_raw_graphs_and_lc_routes(
    RAW_GRAPHS_PATH,
    LC_RESULTS_DIR,
)

route_files = sorted(LC_RESULTS_DIR.glob("graph_*/lc_*_routes_routes.pkl"))
route_lengths = (generated_seed_routes >= 0).sum(dim=-1)
print(f"Graphs loaded:      {len(generated_graphs)}")
print(f"Route files found:  {len(route_files)}")
print(f"Seed routes shape:  {tuple(generated_seed_routes.shape)}")
print(f"Route length min:   {int(route_lengths.min())}")
print(f"Route length max:   {int(route_lengths.max())}")
